# Lab 5 - Classic Spark vs Lakeflow
Repeats the pipeline logic in plain Spark and compares it with the Lakeflow result. The Bronze inputs and the pipeline outputs are first copied into Delta snapshot tables (materialized views have no time travel). The classic code then builds Silver and quarantine from the same Bronze, writes them to `classic_*_lab5` tables and measures its own run time.
The tables are compared as multisets, including Kafka offsets and quarantine reasons, ignoring ingestion timestamps. It compares results on identical stored input, not two live consumers.

In [ ]:
%run ./lab5_00_config

In [ ]:
from time import perf_counter
names = ["titles_bronze", "wiki_bronze", "titles_silver", "wiki_silver", "wiki_quarantine"]
stream_names = ["wiki_bronze", "wiki_silver", "wiki_quarantine"]
versions = {name: delta_version(name) for name in stream_names}
inputs = {}
for name in names:
    source = read_version(name, versions[name]) if name in versions else spark.table(table(name))
    snapshot_name = table("comparison_snapshot_" + name)
    source.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(snapshot_name)
    inputs[name] = spark.table(snapshot_name)
assert inputs["wiki_bronze"].limit(1).count() > 0

started = perf_counter()
classic_titles = valid_rows(clean_titles(inputs["titles_bronze"]), TITLE_RULES)
classic_prepared = annotate_quality(parse_wiki(inputs["wiki_bronze"]), WIKI_RULES)
classic_wiki = classic_prepared.filter("_is_valid").drop("_is_valid", "_quality_errors")
classic_quarantine = classic_prepared.filter("NOT _is_valid")
for name, df in [("titles_silver", classic_titles), ("wiki_silver", classic_wiki), ("wiki_quarantine", classic_quarantine)]:
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table("classic_" + name))
elapsed = perf_counter() - started
keys = ["kafka_topic", "kafka_partition", "kafka_offset"]
assert_equal(spark.table(table("classic_titles_silver")), inputs["titles_silver"], TITLE_BUSINESS_COLUMNS, "classic vs Lakeflow Netflix")
assert_equal(spark.table(table("classic_wiki_silver")), inputs["wiki_silver"], WIKI_BUSINESS_COLUMNS + keys, "classic vs Lakeflow Wikipedia")
assert_equal(spark.table(table("classic_wiki_quarantine")), inputs["wiki_quarantine"], keys + ["_quality_errors"], "classic vs Lakeflow quarantine")
assert versions == {name: delta_version(name) for name in stream_names}, "Pipeline ran during comparison; repeat on a completed update"
for name in ["titles_bronze", "titles_silver"]:
    assert_equal(spark.table(table(name)), inputs[name], inputs[name].columns, f"{name} unchanged during comparison")
print({"frozen_streaming_versions": versions, "classic_transform_and_write_seconds": round(elapsed, 3)})

## Optional: classic Kafka ingestion
Reads the Event Hub with a plain structured stream and its own checkpoint, independent of the pipeline, and writes a classic Bronze table. It is off by default (`run_classic_stream=false`), so a second consumer does not run while the comparison is collected.

In [ ]:
dbutils.widgets.dropdown("run_classic_stream", "false", ["false", "true"])
if dbutils.widgets.get("run_classic_stream") == "true":
    secret = dbutils.secrets.get(scope=secret_scope, key=secret_key)
    options = kafka_options(namespace, eh_name, secret)
    raw = decode_kafka(spark.readStream.format("kafka").options(**options).load())
    query = (raw.writeStream.format("delta").outputMode("append")
             .option("checkpointLocation", f"{volume_path}/checkpoints/classic_wiki_bronze")
             .trigger(availableNow=True).toTable(table("classic_wiki_bronze")))
    query.awaitTermination()
    print("Classic consumer completed; reuse this checkpoint on rerun.")